# Lab ml_02 — k-means, PCA and EM from scratch

**Goals**
1. Implement **Lloyd's k-means** and watch the objective $J$ decrease.
2. Implement **PCA via the SVD** and relate singular values to covariance eigenvalues.
3. Implement **EM for a 1-D Gaussian mixture** (E-step responsibilities, M-step updates) and check the log-likelihood
   never decreases.

Do the paper homework for **k-means & Gaussian mixtures (EM)** first.

**Reading**: Cornell CS4780 lecture notes (https://www.cs.cornell.edu/courses/cs4780/2018fa/lectures/index.html);
Stanford CS229 notes on k-means, EM and PCA (https://cs229.stanford.edu/main_notes.pdf);
Columbia COMS 4771 SVD notes (https://www.cs.columbia.edu/~djhsu/ML/).

## Theory recap

* **k-means** minimises $J = \sum_i \|x_i - c_{z_i}\|^2$ by alternating: assign $z_i = \arg\min_k\|x_i - c_k\|$;
  update $c_k = \text{mean}\{x_i : z_i = k\}$. Neither step increases $J$, so it converges (to a *local* optimum).
* **PCA**: centre $X_c = X - \bar x$; SVD $X_c = U S V^\top$. Rows of $V^\top$ are the principal directions; the variance
  along direction $k$ is $S_k^2/(n-1)$ — the eigenvalues of the sample covariance. Scores: $Z = X_c V_{:,1:k}$.
* **EM for a GMM**: E-step $r_{ik} = \dfrac{\pi_k\,\mathcal N(x_i;\mu_k,\sigma_k^2)}{\sum_j \pi_j\,\mathcal N(x_i;\mu_j,\sigma_j^2)}$;
  M-step with $N_k = \sum_i r_{ik}$: $\pi_k = N_k/n$, $\mu_k = \sum_i r_{ik}x_i/N_k$, $\sigma_k^2 = \sum_i r_{ik}(x_i-\mu_k)^2/N_k$.
  k-means is the "hard" ($r_{ik}\in\{0,1\}$), equal-variance limit.

In [ ]:
import sys, math, re, zlib
from pathlib import Path
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

# make `lc_client` importable whether Jupyter was started in labs/ or in the repo root
LAB_DIR = Path.cwd() if (Path.cwd() / "lc_client.py").exists() else Path.cwd() / "labs"
sys.path.insert(0, str(LAB_DIR))
DATA = LAB_DIR / "data"

rng = np.random.default_rng(0)
CENTERS = np.array([[0, 0], [5, 1], [1, 5]])
Xb = np.vstack([c + 0.8 * rng.normal(size=(80, 2)) for c in CENTERS])
plt.scatter(Xb[:, 0], Xb[:, 1], s=8); plt.axis("equal"); plt.title("three blobs"); plt.show()

## 1. k-means

In [ ]:
def kmeans_objective(X, C, labels):
    """Sum of squared distances of each point to its assigned centroid."""
    # TODO
    raise NotImplementedError


def kmeans(X, k, n_iter=100, seed=0):
    """Lloyd's algorithm. Init: k distinct data points chosen with np.random.default_rng(seed).
    Stop early when the assignments no longer change. If a cluster becomes empty, keep its old centroid.
    Return (centroids (k, d), labels (n,))."""
    # TODO
    raise NotImplementedError

In [ ]:
C, lab = kmeans(Xb, 3, seed=0)
print("objective:", kmeans_objective(Xb, C, lab))
plt.scatter(Xb[:, 0], Xb[:, 1], c=lab, s=8, cmap="viridis"); plt.scatter(C[:, 0], C[:, 1], c="r", marker="x", s=100)
plt.axis("equal"); plt.show()

# Explore: run with seeds 0..9 and k = 1..6; plot the best objective vs k (the "elbow").

## 2. PCA via SVD

In [ ]:
def pca_svd(X, k):
    """Return (components (k, d) unit-norm rows, explained_variance (k,) = S^2/(n-1), Z = Xc @ components.T)."""
    # TODO: centre, np.linalg.svd(Xc, full_matrices=False), take the top-k rows of Vt
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(1)
Xp = rng.normal(size=(400, 3)) @ np.array([[3, 0, 0], [1.5, 1, 0], [0.5, -0.3, 0.3]]) + np.array([2, -1, 0])
comps, ev, Z = pca_svd(Xp, 3)
print("explained variance:", ev, " ratio:", ev / ev.sum())
print("eigenvalues of np.cov:", np.sort(np.linalg.eigvalsh(np.cov(Xp.T)))[::-1])
plt.scatter(Z[:, 0], Z[:, 1], s=6); plt.xlabel("PC1"); plt.ylabel("PC2"); plt.axis("equal"); plt.show()

## 3. EM for a 1-D Gaussian mixture

In [ ]:
def normal_pdf(x, mu, var):
    return np.exp(-(x - mu) ** 2 / (2 * var)) / np.sqrt(2 * np.pi * var)


def e_step(x, pi, mu, var):
    """Responsibilities R[i, k] = P(z_i = k | x_i), shape (n, K)."""
    # TODO: broadcast x[:, None] against the K components, normalise each row
    raise NotImplementedError


def m_step(x, R):
    """Return updated (pi, mu, var)."""
    # TODO (clip variances at 1e-6 to avoid collapse)
    raise NotImplementedError


def log_likelihood(x, pi, mu, var):
    # TODO: sum_i log sum_k pi_k N(x_i; mu_k, var_k)
    raise NotImplementedError


def gmm_em_1d(x, K, n_iter=200, seed=0):
    """Init: means = K distinct data points (np.random.default_rng(seed)), variances = var(x), weights uniform.
    Return (pi, mu, var, loglik_history)."""
    # TODO: alternate E and M steps, recording the log-likelihood
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(2)
z = rng.random(800) < 0.35
xg = np.where(z, rng.normal(-2, 0.7, 800), rng.normal(2.5, 1.2, 800))
pi, mu, var, hist = gmm_em_1d(xg, 2, seed=0)
print("pi", pi, "mu", mu, "sd", np.sqrt(var))

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(hist); ax[0].set_title("log-likelihood per EM iteration")
grid = np.linspace(xg.min(), xg.max(), 300)
ax[1].hist(xg, bins=50, density=True, alpha=0.4)
ax[1].plot(grid, sum(p * normal_pdf(grid, m, v) for p, m, v in zip(pi, mu, var)), "r")
ax[1].set_title("fitted mixture"); plt.show()

*Your observations: what happens with K = 3? With an unlucky seed? Compare k-means on `xg[:, None]`.*

In [ ]:
from lc_client import grade
grade("ml_02_unsupervised", globals())